# Output format

Every max-models code writes its output as one `xarray.Dataset` layout: the grid moments of each species, the fields, marker samples and the provenance of the run. A plot, a diagnostic or a comparison of two codes then reads a run without caring which code produced it. This tutorial builds a small dataset by hand, as a code's writer would, checks it, stores it and reads it back.

In [ ]:
import numpy as np
import plasma_plots
import xarray as xr

import takeout

plasma_plots.set_backend("plotly")
plasma_plots.set_math("latex")

## A run in the format

A 1D run of electrons and ions on 65 nodes, recorded at four times: the densities relax from a bump towards a uniform profile, and the potential follows. The dims are `t`, `species` and the spatial `x` (then `y`, `z`), in that order; every coordinate and variable carries its `units`, SI throughout, and the global attributes say which layout it is (`Conventions`), its frame, the species' mass and charge, and where it came from (`takeout.provenance`).

In [ ]:
t = np.linspace(0.0, 3e-6, 4)
x = np.linspace(0.0, 0.02, 65)
width = 2e-3 * np.sqrt(1.0 + t / 1e-6)
bump = np.exp(-(((x[None, :] - 0.01) / width[:, None]) ** 2)) * (2e-3 / width[:, None])
density = 1e16 * np.stack([1.0 + 0.5 * bump, 1.0 + 0.3 * bump], axis=1)
potential = 5.0 * bump

run = xr.Dataset(
    {
        "density": (
            ("t", "species", "x"),
            density,
            {"units": "1/m^3", "long_name": "$n$"},
        ),
        "potential": (("t", "x"), potential, {"units": "V", "long_name": r"$\phi$"}),
    },
    coords={
        "t": ("t", t, {"units": "s", "long_name": "$t$"}),
        "species": ("species", ["electrons", "ions"]),
        "x": ("x", x, {"units": "m", "long_name": "$x$"}),
    },
    attrs={
        "Conventions": takeout.FORMAT,
        "frame": "cartesian",
        "mass": [9.109e-31, 1.673e-27],
        "charge": [-1.602e-19, 1.602e-19],
        **takeout.provenance("takeout", parameters="cells: 64"),
    },
)
takeout.validate(run)

An empty list: the dataset follows the format. The provenance stamps the code, its version, the git commit of its checkout (with `-dirty` when it has local changes) and the run's configuration:

In [ ]:
{key: run.attrs[key] for key in takeout.PROVENANCE}

`validate` lists every problem it finds instead of stopping at the first; `check` raises a `FormatError` with that list, and `write` refuses to store a dataset that fails it. Densities in the wrong units, a moment index out of order and a missing provenance attribute:

In [ ]:
bad = run.copy()
bad["density"] = bad["density"].assign_attrs(units="1/cm^3")
bad["moment_yx"] = bad["density"].assign_attrs(units="1/(m s^2)")
del bad.attrs["git_hash"]
takeout.validate(bad)

## Plots from the format

Since the names, dims and units are fixed, plasma-plots draws the run without being told what it is: the density profiles of each species at every recorded time, one figure per species.

In [ ]:
for species in run.species.values:
    with plasma_plots.figure() as fig:
        run.density.sel(species=species).plasma.plot.profiles(
            x="x", over="t", title=str(species), ax=fig[0]
        )

## Marker samples

A few markers of each species can be stored with the moments, over a `sample` dim: `marker_samples` turns positions, velocities, weights and ids into the `sample_*` variables, padding with `sample_id = -1` and NaN when a species has fewer markers than the dim is long.

In [ ]:
rng = np.random.default_rng(0)
samples = takeout.marker_samples(
    ids=np.arange(3),
    positions=rng.uniform(0.0, 0.02, (1, 3)),
    velocities=rng.normal(0.0, 1e5, (3, 3)),
    weights=np.full(3, 1e8),
    num_samples=4,
)
samples[["sample_id", "sample_x", "sample_weight"]].to_dataframe()

## Files

`write` checks the dataset and stores it as netCDF; `read` loads it and checks it again, so a file that does not follow the format is noticed when it is opened, not in a plot later.

In [ ]:
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as folder:
    path = takeout.write(run, Path(folder) / "run.nc")
    back = takeout.read(path)
xr.testing.assert_allclose(back, run)
back.attrs["code"], back.attrs["created"]